In [11]:
# ADVANCED GAME AI AGENT (FULL UPGRADED PROJECT)
# Includes:
# - Tic Tac Toe (Minimax + Alpha-Beta + GUI Fix)
# - Mini Chess (6x6 with basic rules + AI)
# - Snake AI (A* Pathfinding)
# - Smooth, colorful, non-laggy GUI

import tkinter as tk
from tkinter import messagebox
import random
import heapq

# ================= MAIN MENU =================
class GameMenu:
    def __init__(self, root):
        self.root = root
        root.title("AI Game Hub")
        root.geometry("500x500")
        root.configure(bg="#0f172a")

        tk.Label(root, text="AI GAME HUB", font=("Arial", 26, "bold"), fg="#22c55e", bg="#0f172a").pack(pady=30)

        self.make_btn("Tic Tac Toe", self.open_ttt)
        self.make_btn("Mini Chess", self.open_chess)
        self.make_btn("Snake AI", self.open_snake)

    def make_btn(self, text, cmd):
        tk.Button(self.root, text=text, command=cmd,
                  font=("Arial", 14), width=20,
                  bg="#2563eb", fg="white",
                  activebackground="#1d4ed8").pack(pady=10)

    def open_ttt(self): TicTacToe()
    def open_chess(self): MiniChess()
    def open_snake(self): SnakeAI()

# ================= TIC TAC TOE =================
class TicTacToe:
    def __init__(self):
        self.win = tk.Toplevel()
        self.win.title("Tic Tac Toe AI")
        self.win.configure(bg="#0b1220")

        self.board = [""]*9
        self.buttons = []
        self.game_over = False
        self.scores = {"X":0, "O":0, "D":0}
        self.difficulty = tk.StringVar(value="Hard")  # Easy/Medium/Hard

        top = tk.Frame(self.win, bg="#0b1220")
        top.grid(row=0, column=0, columnspan=3, sticky="ew")

        self.score_lbl = tk.Label(top, text=self.score_text(), fg="#e5e7eb", bg="#0b1220", font=("Arial", 12, "bold"))
        self.score_lbl.pack(side="left", padx=10, pady=6)

        diff_menu = tk.OptionMenu(top, self.difficulty, "Easy", "Medium", "Hard")
        diff_menu.config(bg="#1f2937", fg="white", highlightthickness=0)
        diff_menu.pack(side="right", padx=10)

        board_frame = tk.Frame(self.win, bg="#0b1220")
        board_frame.grid(row=1, column=0, columnspan=3)

        for i in range(9):
            btn = tk.Button(board_frame, text="", font=("Arial", 22, "bold"), width=5, height=2,
                            bg="#111827", fg="#e5e7eb", activebackground="#374151",
                            command=lambda i=i: self.move(i))
            btn.grid(row=i//3, column=i%3, padx=4, pady=4)
            self.buttons.append(btn)

        ctrl = tk.Frame(self.win, bg="#0b1220")
        ctrl.grid(row=2, column=0, columnspan=3, pady=8)

        tk.Button(ctrl, text="Restart", command=self.reset_board, bg="#2563eb", fg="white").pack(side="left", padx=6)
        tk.Button(ctrl, text="Reset Scores", command=self.reset_scores, bg="#ef4444", fg="white").pack(side="left", padx=6)

    def score_text(self):
        return f"You (X): {self.scores['X']}   AI (O): {self.scores['O']}   Draws: {self.scores['D']}"

    def update_score_label(self):
        self.score_lbl.config(text=self.score_text())

    def move(self, i):
        if self.board[i] == "" and not self.game_over:
            self.place(i, "X")

            win_line = self.get_win_line(self.board, "X")
            if win_line:
                self.scores['X'] += 1
                self.end_game("You Win!", win_line)
                return

            if "" not in self.board:
                self.scores['D'] += 1
                self.end_game("Draw")
                return

            self.win.after(120, self.ai)  # slight delay for UX

    def ai(self):
        if self.game_over: return

        move = None
        level = self.difficulty.get()

        if level == "Easy":
            empty = [i for i,v in enumerate(self.board) if v==""]
            move = random.choice(empty) if empty else None
        elif level == "Medium":
            # 70% optimal, 30% random → beatable
            if random.random() < 0.3:
                empty = [i for i,v in enumerate(self.board) if v==""]
                move = random.choice(empty) if empty else None
            else:
                _, move = self.minimax(self.board, True, -999, 999)
        else:  # Hard
            # Mostly optimal but slightly imperfect to allow rare wins
            if random.random() < 0.15:
                empty = [i for i,v in enumerate(self.board) if v==""]
                move = random.choice(empty) if empty else None
            else:
                _, move = self.minimax(self.board, True, -999, 999)

        if move is not None:
            self.place(move, "O")

            win_line = self.get_win_line(self.board, "O")
            if win_line:
                self.scores['O'] += 1
                self.end_game("AI Wins", win_line)
                return

            if "" not in self.board:
                self.scores['D'] += 1
                self.end_game("Draw")

    def place(self, i, p):
        self.board[i] = p
        color = "#22c55e" if p=="X" else "#f87171"
        self.buttons[i].config(text=p, bg=color)

    def minimax(self, board, is_max, alpha, beta):
        if self.check_winner(board, "O"): return (1, None)
        if self.check_winner(board, "X"): return (-1, None)
        if "" not in board: return (0, None)

        best = (-999, None) if is_max else (999, None)

        for i in range(9):
            if board[i] == "":
                board[i] = "O" if is_max else "X"
                score = self.minimax(board, not is_max, alpha, beta)[0]
                board[i] = ""

                if is_max:
                    if score > best[0]: best = (score, i)
                    alpha = max(alpha, score)
                else:
                    if score < best[0]: best = (score, i)
                    beta = min(beta, score)

                if beta <= alpha: break

        return best

    def check_winner(self, board, player):
        return self.get_win_line(board, player) is not None

    def get_win_line(self, board, player):
        wins = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
        for w in wins:
            if all(board[i] == player for i in w):
                return w
        return None

    def highlight(self, line):
        for i in line:
            self.buttons[i].config(bg="#fbbf24")

    def end_game(self, text, line=None):
        self.game_over = True
        if line:
            self.highlight(line)
        self.update_score_label()
        self.win.after(100, lambda: messagebox.showinfo("Game Over", text))

    def reset_board(self):
        self.board = [""]*9
        self.game_over = False
        for b in self.buttons:
            b.config(text="", bg="#111827")

    def reset_scores(self):
        self.scores = {"X":0, "O":0, "D":0}
        self.update_score_label()

# ================= MINI CHESS (ADVANCED VERSION) =================
class MiniChess:
    def __init__(self):
        self.win = tk.Toplevel()
        self.win.title("Advanced Mini Chess AI")
        self.size = 6
        self.selected = None
        self.turn = 'P'

        self.board = [
            ['r','n','b','q','k','r'],
            ['p','p','p','p','p','p'],
            ['','','','','',''],
            ['','','','','',''],
            ['P','P','P','P','P','P'],
            ['R','N','B','Q','K','R']
        ]

        self.buttons = [[None]*6 for _ in range(6)]
        self.highlighted = []
        self.draw()

    def draw(self):
        for r in range(6):
            for c in range(6):
                color = "#f0d9b5" if (r+c)%2==0 else "#b58863"
                btn = tk.Button(self.win, text=self.board[r][c], width=6, height=3,
                                font=("Arial", 18, "bold"),
                                bg=color, command=lambda r=r,c=c: self.click(r,c))
                btn.grid(row=r, column=c)
                self.buttons[r][c] = btn

    def refresh(self):
        for r in range(6):
            for c in range(6):
                base = "#f0d9b5" if (r+c)%2==0 else "#b58863"
                self.buttons[r][c].config(text=self.board[r][c], bg=base)

        # highlight check
        if self.is_in_check('P'):
            self.highlight_king('P')
        if self.is_in_check('p'):
            self.highlight_king('p')

    def highlight_king(self, player):
        king = 'K' if player=='P' else 'k'
        for r in range(6):
            for c in range(6):
                if self.board[r][c] == king:
                    self.buttons[r][c].config(bg="#ef4444")

    def click(self, r, c):
        if self.turn != 'P': return
        self.clear_highlights()

        if self.selected:
            if (r,c) in self.get_valid_moves(self.selected):
                self.make_move(self.selected, (r,c))

                if self.is_checkmate('p'):
                    messagebox.showinfo("Game Over", "You Win! Checkmate!")
                    self.win.destroy()
                    return

                self.selected = None
                self.turn = 'p'
                self.win.after(200, self.ai_move)
            else:
                self.selected = None
        else:
            if self.board[r][c].isupper():
                self.selected = (r,c)
                self.show_moves(r,c)

    def show_moves(self, r, c):
        moves = self.get_valid_moves((r,c))
        for (mr,mc) in moves:
            self.buttons[mr][mc].config(bg="#34d399")
            self.highlighted.append((mr,mc))
        self.buttons[r][c].config(bg="#60a5fa")

    def clear_highlights(self):
        for (r,c) in self.highlighted:
            base = "#f0d9b5" if (r+c)%2==0 else "#b58863"
            self.buttons[r][c].config(bg=base)
        self.highlighted = []
        self.refresh()

    # ================= CORE LOGIC =================

    def get_valid_moves(self, pos):
        r,c = pos
        piece = self.board[r][c]
        player = 'P' if piece.isupper() else 'p'
        moves = []

        for r2 in range(6):
            for c2 in range(6):
                if self.valid_move((r,c),(r2,c2),player):
                    temp = [row[:] for row in self.board]

                    self.board[r2][c2] = self.board[r][c]
                    self.board[r][c] = ''

                    if not self.is_in_check(player):
                        moves.append((r2,c2))

                    self.board = temp
        return moves

    def is_in_check(self, player):
        king = 'K' if player=='P' else 'k'
        king_pos = None

        for r in range(6):
            for c in range(6):
                if self.board[r][c] == king:
                    king_pos = (r,c)

        for r in range(6):
            for c in range(6):
                piece = self.board[r][c]
                if piece and ((player=='P' and piece.islower()) or (player=='p' and piece.isupper())):
                    if self.valid_move((r,c), king_pos, 'p' if player=='P' else 'P'):
                        return True
        return False

    def is_checkmate(self, player):
        if not self.is_in_check(player):
            return False

        for r in range(6):
            for c in range(6):
                piece = self.board[r][c]
                if piece and ((player=='P' and piece.isupper()) or (player=='p' and piece.islower())):
                    if self.get_valid_moves((r,c)):
                        return False
        return True

    def make_move(self, start, end):
        sr,sc = start
        er,ec = end
        self.board[er][ec] = self.board[sr][sc]
        self.board[sr][sc] = ''
        self.refresh()

    # ================= MOVEMENT =================

    def valid_move(self, start, end, player):
        sr,sc = start
        er,ec = end
        piece = self.board[sr][sc]

        if not piece: return False
        if player=='P' and not piece.isupper(): return False
        if player=='p' and not piece.islower(): return False

        target = self.board[er][ec]
        if target and (target.isupper()==piece.isupper()): return False

        dr, dc = er-sr, ec-sc

        if piece.lower()=='p':
            direction = -1 if piece.isupper() else 1
            if dc==0 and dr==direction and target=='': return True
            if abs(dc)==1 and dr==direction and target!='': return True

        if piece.lower()=='r':
            if sr==er:
                step = 1 if ec>sc else -1
                for c in range(sc+step, ec, step):
                    if self.board[sr][c] != '': return False
                return True
            if sc==ec:
                step = 1 if er>sr else -1
                for r in range(sr+step, er, step):
                    if self.board[r][sc] != '': return False
                return True

        if piece.lower()=='n':
            return (abs(dr),abs(dc)) in [(2,1),(1,2)]

        if piece.lower()=='b':
            if abs(dr)==abs(dc):
                step_r = 1 if dr>0 else -1
                step_c = 1 if dc>0 else -1
                for i in range(1,abs(dr)):
                    if self.board[sr+i*step_r][sc+i*step_c] != '': return False
                return True

        if piece.lower()=='q':
            if sr==er or sc==ec or abs(dr)==abs(dc):
                step_r = 0 if dr==0 else (1 if dr>0 else -1)
                step_c = 0 if dc==0 else (1 if dc>0 else -1)
                i=1
                while (sr+i*step_r, sc+i*step_c)!=(er,ec):
                    if self.board[sr+i*step_r][sc+i*step_c] != '': return False
                    i+=1
                return True

        if piece.lower()=='k':
            return abs(dr)<=1 and abs(dc)<=1

        return False

    # ================= AI =================

    def ai_move(self):
        best_score = -9999
        best_move = None

        for r in range(6):
            for c in range(6):
                if self.board[r][c].islower():
                    for move in self.get_valid_moves((r,c)):
                        temp = [row[:] for row in self.board]
                        self.make_temp_move((r,c), move)
                        score = self.minimax(2, False)
                        self.board = temp

                        if score > best_score:
                            best_score = score
                            best_move = ((r,c), move)

        if best_move:
            self.make_move(*best_move)

            if self.is_checkmate('P'):
                messagebox.showinfo("Game Over", "AI Wins! Checkmate!")
                self.win.destroy()
                return

        self.turn = 'P'

    def minimax(self, depth, is_max):
        if depth == 0:
            return self.evaluate()

        if is_max:
            best = -9999
            for r in range(6):
                for c in range(6):
                    if self.board[r][c].islower():
                        for move in self.get_valid_moves((r,c)):
                            temp = [row[:] for row in self.board]
                            self.make_temp_move((r,c), move)
                            best = max(best, self.minimax(depth-1, False))
                            self.board = temp
            return best
        else:
            best = 9999
            for r in range(6):
                for c in range(6):
                    if self.board[r][c].isupper():
                        for move in self.get_valid_moves((r,c)):
                            temp = [row[:] for row in self.board]
                            self.make_temp_move((r,c), move)
                            best = min(best, self.minimax(depth-1, True))
                            self.board = temp
            return best

    def make_temp_move(self, start, end):
        sr,sc = start
        er,ec = end
        self.board[er][ec] = self.board[sr][sc]
        self.board[sr][sc] = ''
        
    def evaluate(self):
        values = {'p':1,'n':3,'b':3,'r':5,'q':9,'k':50}
        score = 0
        for row in self.board:
            for p in row:
                if p:
                    val = values[p.lower()]
                    score += val if p.islower() else -val
        return score
# ================= SNAKE AI ================= ================= ================= (A*) =================
class SnakeAI:
    def __init__(self):
        self.win = tk.Toplevel()
        self.win.title("Snake AI")
        self.canvas = tk.Canvas(self.win, width=400, height=400, bg="black")
        self.canvas.pack()

        self.snake = [(100,100),(90,100),(80,100)]
        self.food = (200,200)

        self.run()

    def run(self):
        self.move()
        self.draw()
        self.win.after(80, self.run)

    def move(self):
        path = self.a_star(self.snake[0], self.food)
        if len(path) > 1:
            new_head = path[1]
        else:
            new_head = (self.snake[0][0]+10, self.snake[0][1])

        self.snake.insert(0, new_head)

        if new_head == self.food:
            self.food = (random.randint(0,39)*10, random.randint(0,39)*10)
        else:
            self.snake.pop()

    def a_star(self, start, goal):
        open_set = [(0, start)]
        came = {}
        g = {start:0}

        while open_set:
            _, current = heapq.heappop(open_set)
            if current == goal:
                path = []
                while current in came:
                    path.append(current)
                    current = came[current]
                path.append(start)
                return path[::-1]

            for dx,dy in [(10,0),(-10,0),(0,10),(0,-10)]:
                nxt = (current[0]+dx, current[1]+dy)
                if 0<=nxt[0]<400 and 0<=nxt[1]<400:
                    temp = g[current]+1
                    if nxt not in g or temp < g[nxt]:
                        g[nxt] = temp
                        f = temp + abs(nxt[0]-goal[0]) + abs(nxt[1]-goal[1])
                        heapq.heappush(open_set, (f, nxt))
                        came[nxt] = current
        return [start]

    def draw(self):
        self.canvas.delete("all")
        for x,y in self.snake:
            self.canvas.create_rectangle(x,y,x+10,y+10, fill="#22c55e")
        self.canvas.create_rectangle(self.food[0], self.food[1], self.food[0]+10, self.food[1]+10, fill="#ef4444")

# ================= RUN =================
root = tk.Tk()
GameMenu(root)
root.mainloop()


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'matplotlib', 'networkx', 'numpy', 'ipywidgets'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import heapq, math, warnings
from collections import deque
warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = '#f5f5f5'
plt.rcParams['axes.facecolor']   = '#f5f5f5'

# ══════════════════════════════════════════════════════════════
#  GRAPH DATA
# ══════════════════════════════════════════════════════════════

EDGES = [
    ('A','B',4),('A','C',3),('B','D',2),('B','E',5),
    ('C','E',1),('C','F',6),('D','G',3),('E','G',4),
    ('E','H',2),('F','H',3),('G','I',2),('G','J',5),
    ('H','I',1),('I','J',3)
]
POS = {
    'A':(0,2),'B':(1,3),'C':(1,1),'D':(2,4),
    'E':(2,2),'F':(2,0),'G':(3,3.5),'H':(3,1),
    'I':(4,2.5),'J':(5,3)
}

def heuristic(node, goal):
    x1,y1=POS[node]; x2,y2=POS[goal]
    return math.sqrt((x2-x1)**2+(y2-y1)**2)

def build_graph(edges):
    g={}
    for u,v,w in edges:
        g.setdefault(u,[]).append((v,w))
        g.setdefault(v,[]).append((u,w))
    return g

GRAPH = build_graph(EDGES)

# ══════════════════════════════════════════════════════════════
#  DRAW HELPER
# ══════════════════════════════════════════════════════════════

COLORS = {
    'start':'#D85A30','goal':'#7F77DD','path':'#E24B4A',
    'visited':'#1D9E75','frontier':'#EF9F27','default':'#378ADD'
}

def draw_graph(ax, visited=(), frontier=(), path=(), start=None, goal=None, title=''):
    G=nx.Graph()
    for u,v,w in EDGES: G.add_edge(u,v,weight=w)
    visited=set(visited); frontier=set(frontier); path=list(path)
    path_edges={(path[i],path[i+1]) for i in range(len(path)-1)}
    path_edges|={(b,a) for a,b in path_edges}
    node_colors=[
        COLORS['start']    if n==start    else
        COLORS['goal']     if n==goal     else
        COLORS['path']     if n in path   else
        COLORS['visited']  if n in visited  else
        COLORS['frontier'] if n in frontier else
        COLORS['default']
        for n in G.nodes()
    ]
    edge_colors=['#E24B4A' if (u,v) in path_edges else '#cccccc' for u,v in G.edges()]
    edge_widths=[3.0       if (u,v) in path_edges else 1.2       for u,v in G.edges()]
    nx.draw_networkx(G,pos=POS,ax=ax,node_color=node_colors,node_size=700,
                     edge_color=edge_colors,width=edge_widths,
                     font_color='white',font_size=11,font_weight='bold',with_labels=True)
    nx.draw_networkx_edge_labels(G,pos=POS,edge_labels=nx.get_edge_attributes(G,'weight'),
                                  ax=ax,font_size=8,font_color='#555')
    ax.set_title(title,fontsize=10,fontweight='bold')
    ax.axis('off')

# ══════════════════════════════════════════════════════════════
#  SEARCH ALGORITHMS
# ══════════════════════════════════════════════════════════════

def bfs(graph, start, goal):
    queue=deque([(start,[start])]); visited=set([start]); steps=[]
    while queue:
        node,path=queue.popleft()
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in queue},path=[]))
        if node==goal: steps[-1]['path']=path; return path,steps
        for nb,_ in sorted(graph.get(node,[])):
            if nb not in visited:
                visited.add(nb); queue.append((nb,path+[nb]))
    return None,steps

def dfs(graph, start, goal):
    stack=[(start,[start])]; visited=set(); steps=[]
    while stack:
        node,path=stack.pop()
        if node in visited: continue
        visited.add(node)
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in stack},path=[]))
        if node==goal: steps[-1]['path']=path; return path,steps
        for nb,_ in sorted(graph.get(node,[]),reverse=True):
            if nb not in visited: stack.append((nb,path+[nb]))
    return None,steps

def ucs(graph, start, goal):
    heap=[(0,start,[start])]; visited={}; steps=[]
    while heap:
        cost,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=cost
        steps.append(dict(cur=node,cost=cost,visited=dict(visited),
                          frontier=[(c,n) for c,n,_ in heap],path=[]))
        if node==goal: steps[-1]['path']=path; return path,cost,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited: heapq.heappush(heap,(cost+w,nb,path+[nb]))
    return None,float('inf'),steps

def astar(graph, start, goal):
    h0=heuristic(start,goal)
    heap=[(h0,0,start,[start])]; visited={}; steps=[]
    while heap:
        f,g,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=g
        steps.append(dict(cur=node,g=g,h=round(heuristic(node,goal),2),
                          f=round(f,2),visited=dict(visited),
                          frontier=[(fc,n) for fc,_,n,_ in heap],path=[]))
        if node==goal: steps[-1]['path']=path; return path,g,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited:
                g2=g+w; f2=g2+heuristic(nb,goal)
                heapq.heappush(heap,(f2,g2,nb,path+[nb]))
    return None,float('inf'),steps

# ══════════════════════════════════════════════════════════════
#  UI
# ══════════════════════════════════════════════════════════════

display(HTML("""
<div style='background:linear-gradient(135deg,#1a237e,#283593);
             color:white;padding:18px 26px;border-radius:12px;margin-bottom:14px'>
  <h2 style='margin:0;font-size:20px'>Graph Search Algorithm Visualizer</h2>
  <p style='margin:5px 0 0;opacity:.85;font-size:13px'>
    BFS &nbsp;|&nbsp; DFS &nbsp;|&nbsp; UCS &nbsp;|&nbsp; A*
  </p>
</div>
"""))

algo_toggle = widgets.ToggleButtons(
    options=['BFS','DFS','UCS','A*'],
    button_style='info',
    style={'button_width':'110px'}
)
node_labels = sorted(GRAPH.keys())
start_dd = widgets.Dropdown(options=node_labels, value='A', description='Start:',
                            style={'description_width':'45px'},
                            layout=widgets.Layout(width='140px'))
goal_dd  = widgets.Dropdown(options=node_labels, value='J', description='Goal:',
                            style={'description_width':'45px'},
                            layout=widgets.Layout(width='140px'))
run_btn  = widgets.Button(description='Run', button_style='success',
                          layout=widgets.Layout(width='90px'))

graph_out = widgets.Output()
log_out   = widgets.Output()

with graph_out:
    fig,ax=plt.subplots(figsize=(10,5))
    draw_graph(ax, start='A', goal='J', title='Pick start & goal nodes, choose an algorithm, then click Run')
    plt.tight_layout(); plt.show()

def run_search(b):
    s,g  = start_dd.value, goal_dd.value
    algo = algo_toggle.value
    if s==g:
        with log_out: clear_output(); print('Start and goal must be different nodes.')
        return

    cost_str=''; is_cost=False
    if   algo=='BFS': path,steps=bfs(GRAPH,s,g)
    elif algo=='DFS': path,steps=dfs(GRAPH,s,g)
    elif algo=='UCS': path,cost,steps=ucs(GRAPH,s,g);   cost_str=f'  |  Cost: {cost}';         is_cost=True
    else:             path,cost,steps=astar(GRAPH,s,g); cost_str=f'  |  Cost: {round(cost,2)}'; is_cost=True

    with graph_out:
        clear_output(wait=True)
        n=len(steps)
        idxs=list(dict.fromkeys([int(i*(n-1)/7) for i in range(7)]+[n-1])) if n>=8 else list(range(n))
        idxs=idxs[:8]
        cols=4; rows=max(1,math.ceil(len(idxs)/cols))
        fig,axes=plt.subplots(rows,cols,figsize=(16,rows*4.2))
        axes=np.array(axes).flatten()
        for ax in axes: ax.axis('off')
        for i,idx in enumerate(idxs):
            st=steps[idx]
            vis=set(st['visited'].keys()) if isinstance(st['visited'],dict) else st['visited']
            fr =[nn for _,nn in st['frontier']] if is_cost else st['frontier']
            extra=''
            if 'f' in st:      extra=f'  f={st["f"]}  g={st["g"]}  h={st["h"]}'
            elif 'cost' in st: extra=f'  cost={st["cost"]}'
            draw_graph(axes[i],visited=vis,frontier=fr,
                       path=st.get('path',[]),start=s,goal=g,
                       title=f'Step {idx+1} - {st["cur"]}'+extra)
        path_str=' -> '.join(path) if path else 'None'
        plt.suptitle(f'{algo}:  {s} -> {g}  |  Path: {path_str}{cost_str}  |  Nodes expanded: {len(steps)}',
                     fontsize=12,fontweight='bold',y=1.01)
        plt.tight_layout(); plt.show()

    with log_out:
        clear_output(wait=True)
        sep='─'*66
        print(sep)
        print(f'{algo} Traversal Log  |  {s} -> {g}')
        print(sep)
        for i,st in enumerate(steps):
            tag=' <- GOAL' if st.get('path') else ''
            if 'f' in st:
                vis_str=', '.join(f'{k}(g={v})' for k,v in sorted(st['visited'].items()))
                print(f'Step {i+1:2d}: {st["cur"]}  f={st["f"]:5.2f}  g={st["g"]}  h={st["h"]:4.2f}  |  {vis_str}{tag}')
            elif 'cost' in st:
                vis_str=', '.join(f'{k}({v})' for k,v in sorted(st['visited'].items()))
                print(f'Step {i+1:2d}: {st["cur"]}  cost={st["cost"]:2d}  |  {vis_str}{tag}')
            else:
                print(f'Step {i+1:2d}: Visit {st["cur"]}  |  Visited: {{", ".join(sorted(st["visited"]))}}  {tag}')
        print(sep)
        print(f'Path: {" -> ".join(path) if path else "No path found"}{cost_str}')

run_btn.on_click(run_search)

display(HTML('<b>Algorithm:</b>'))
display(algo_toggle)
display(widgets.HBox([start_dd, goal_dd, run_btn],
                     layout=widgets.Layout(gap='10px', margin='8px 0')))
display(HTML("""
<div style='display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 12px;font-size:13px'>
  <b style='background:#D85A30;padding:2px 10px;border-radius:4px;color:white'>Start</b>
  <b style='background:#7F77DD;padding:2px 10px;border-radius:4px;color:white'>Goal</b>
  <b style='background:#EF9F27;padding:2px 10px;border-radius:4px;color:white'>Frontier</b>
  <b style='background:#1D9E75;padding:2px 10px;border-radius:4px;color:white'>Visited</b>
  <b style='background:#E24B4A;padding:2px 10px;border-radius:4px;color:white'>Final Path</b>
  <b style='background:#378ADD;padding:2px 10px;border-radius:4px;color:white'>Unvisited</b>
</div>
"""))
display(graph_out)
display(HTML('<b style="font-size:13px">Traversal Log:</b>'))
display(log_out)
display(HTML("""
<div style='background:#1e3a5f;border-left:4px solid #5c9bd6;
            padding:10px 14px;border-radius:6px;margin-top:12px;font-size:13px;color:#e8f0fe'>
  <b style='color:#ffffff'>BFS</b> <span style='color:#e8f0fe'>— Queue, explores level by level, fewest hops guaranteed</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>DFS</b> <span style='color:#e8f0fe'>— Stack, dives deep first, memory efficient, not always shortest</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>UCS</b> <span style='color:#e8f0fe'>— Min-heap on g(n), expands lowest cumulative cost, weighted optimal</span> &nbsp;|&nbsp;
  <b style='color:#ffffff'>A*</b>  <span style='color:#e8f0fe'>— Min-heap on f(n)=g(n)+h(n), heuristic-guided, faster than UCS</span>
</div>
"""))

ToggleButtons(button_style='info', options=('BFS', 'DFS', 'UCS', 'A*'), style=ToggleButtonsStyle(button_width=…

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1000x500 with 1 Axes>', '…

Output()

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'matplotlib', 'networkx', 'numpy', 'ipywidgets'],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import heapq, math, warnings
from collections import deque
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#f5f5f5'
plt.rcParams['axes.facecolor']   = '#f5f5f5'

# ============================
# HEADER (PURE PYTHON)
# ============================
print("\n" + "="*60)
print("      GRAPH SEARCH ALGORITHM VISUALIZER")
print("         BFS | DFS | UCS | A*")
print("="*60 + "\n")

# ============================
# GRAPH DATA
# ============================

EDGES = [
    ('A','B',4),('A','C',3),('B','D',2),('B','E',5),
    ('C','E',1),('C','F',6),('D','G',3),('E','G',4),
    ('E','H',2),('F','H',3),('G','I',2),('G','J',5),
    ('H','I',1),('I','J',3)
]

POS = {
    'A':(0,2),'B':(1,3),'C':(1,1),'D':(2,4),
    'E':(2,2),'F':(2,0),'G':(3,3.5),'H':(3,1),
    'I':(4,2.5),'J':(5,3)
}

def heuristic(node, goal):
    x1,y1=POS[node]; x2,y2=POS[goal]
    return math.sqrt((x2-x1)**2+(y2-y1)**2)

def build_graph(edges):
    g={}
    for u,v,w in edges:
        g.setdefault(u,[]).append((v,w))
        g.setdefault(v,[]).append((u,w))
    return g

GRAPH = build_graph(EDGES)

# ============================
# DRAW
# ============================

COLORS = {
    'start':'#D85A30','goal':'#7F77DD','path':'#E24B4A',
    'visited':'#1D9E75','frontier':'#EF9F27','default':'#378ADD'
}

def draw_graph(ax, visited=(), frontier=(), path=(), start=None, goal=None, title=''):
    G=nx.Graph()
    for u,v,w in EDGES: G.add_edge(u,v,weight=w)

    visited=set(visited); frontier=set(frontier); path=list(path)
    path_edges={(path[i],path[i+1]) for i in range(len(path)-1)}
    path_edges|={(b,a) for a,b in path_edges}

    node_colors=[
        COLORS['start'] if n==start else
        COLORS['goal'] if n==goal else
        COLORS['path'] if n in path else
        COLORS['visited'] if n in visited else
        COLORS['frontier'] if n in frontier else
        COLORS['default']
        for n in G.nodes()
    ]

    edge_colors=['#E24B4A' if (u,v) in path_edges else '#cccccc' for u,v in G.edges()]
    edge_widths=[3.0 if (u,v) in path_edges else 1.2 for u,v in G.edges()]

    nx.draw_networkx(G,pos=POS,ax=ax,node_color=node_colors,node_size=700,
                     edge_color=edge_colors,width=edge_widths,
                     font_color='white',font_size=11,font_weight='bold')

    nx.draw_networkx_edge_labels(G,pos=POS,
        edge_labels=nx.get_edge_attributes(G,'weight'),
        ax=ax,font_size=8)

    ax.set_title(title,fontsize=10)
    ax.axis('off')

# ============================
# ALGORITHMS
# ============================

def bfs(graph, start, goal):
    queue=deque([(start,[start])]); visited=set([start]); steps=[]
    while queue:
        node,path=queue.popleft()
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in queue},path=[]))
        if node==goal:
            steps[-1]['path']=path
            return path,steps
        for nb,_ in sorted(graph.get(node,[])):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb,path+[nb]))
    return None,steps

def dfs(graph, start, goal):
    stack=[(start,[start])]; visited=set(); steps=[]
    while stack:
        node,path=stack.pop()
        if node in visited: continue
        visited.add(node)
        steps.append(dict(cur=node,visited=set(visited),
                          frontier={n for n,_ in stack},path=[]))
        if node==goal:
            steps[-1]['path']=path
            return path,steps
        for nb,_ in sorted(graph.get(node,[]),reverse=True):
            if nb not in visited:
                stack.append((nb,path+[nb]))
    return None,steps

def ucs(graph, start, goal):
    heap=[(0,start,[start])]; visited={}; steps=[]
    while heap:
        cost,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=cost
        steps.append(dict(cur=node,cost=cost,visited=dict(visited),
                          frontier=[(c,n) for c,n,_ in heap],path=[]))
        if node==goal:
            steps[-1]['path']=path
            return path,cost,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited:
                heapq.heappush(heap,(cost+w,nb,path+[nb]))
    return None,float('inf'),steps

def astar(graph, start, goal):
    heap=[(heuristic(start,goal),0,start,[start])]
    visited={}; steps=[]
    while heap:
        f,g,node,path=heapq.heappop(heap)
        if node in visited: continue
        visited[node]=g
        steps.append(dict(cur=node,g=g,
                          h=round(heuristic(node,goal),2),
                          f=round(f,2),
                          visited=dict(visited),
                          frontier=[(fc,n) for fc,_,n,_ in heap],
                          path=[]))
        if node==goal:
            steps[-1]['path']=path
            return path,g,steps
        for nb,w in sorted(graph.get(node,[])):
            if nb not in visited:
                g2=g+w
                f2=g2+heuristic(nb,goal)
                heapq.heappush(heap,(f2,g2,nb,path+[nb]))
    return None,float('inf'),steps

# ============================
# UI (NO HTML)
# ============================

print("Select Algorithm:")

algo_toggle = widgets.ToggleButtons(
    options=['BFS','DFS','UCS','A*'],
    button_style='info'
)

node_labels = sorted(GRAPH.keys())

start_dd = widgets.Dropdown(options=node_labels, value='A', description='Start:')
goal_dd  = widgets.Dropdown(options=node_labels, value='J', description='Goal:')
run_btn  = widgets.Button(description='Run', button_style='success')

graph_out = widgets.Output()
log_out   = widgets.Output()

with graph_out:
    fig,ax=plt.subplots(figsize=(10,5))
    draw_graph(ax,start='A',goal='J',
               title='Pick start & goal nodes and click Run')
    plt.show()

def run_search(b):
    s,g= start_dd.value, goal_dd.value
    algo=algo_toggle.value

    if s==g:
        print("Start and goal must be different.")
        return

    if algo=='BFS':
        path,steps=bfs(GRAPH,s,g)
    elif algo=='DFS':
        path,steps=dfs(GRAPH,s,g)
    elif algo=='UCS':
        path,cost,steps=ucs(GRAPH,s,g)
    else:
        path,cost,steps=astar(GRAPH,s,g)

    with graph_out:
        clear_output()
        fig,ax=plt.subplots(figsize=(10,5))
        draw_graph(ax,path=path,start=s,goal=g,
                   title=f"{algo} Path: {' -> '.join(path)}")
        plt.show()

    with log_out:
        clear_output()
        print("\nTraversal Log:\n")
        for i,st in enumerate(steps):
            print(f"Step {i+1}: {st['cur']}")

run_btn.on_click(run_search)

# ============================
# DISPLAY
# ============================

display(algo_toggle)
display(start_dd)
display(goal_dd)
display(run_btn)

print("\nLegend:")
print("Start (Red), Goal (Purple), Path (Light Red),")
print("Visited (Green), Frontier (Orange), Unvisited (Blue)\n")

display(graph_out)

print("\nTraversal Log Output:")
display(log_out)

print("\nAlgorithm Info:")
print("BFS — Level-wise search")
print("DFS — Depth-first search")
print("UCS — Cost-based optimal search")
print("A* — Heuristic + cost optimal search")


      GRAPH SEARCH ALGORITHM VISUALIZER
         BFS | DFS | UCS | A*

Select Algorithm:


ToggleButtons(button_style='info', options=('BFS', 'DFS', 'UCS', 'A*'), value='BFS')

Dropdown(description='Start:', options=('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'), value='A')

Dropdown(description='Goal:', index=9, options=('A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'), value='J')

Button(button_style='success', description='Run', style=ButtonStyle())


Legend:
Start (Red), Goal (Purple), Path (Light Red),
Visited (Green), Frontier (Orange), Unvisited (Blue)



Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1000x500 with 1 Axes>', '…


Traversal Log Output:


Output()


Algorithm Info:
BFS — Level-wise search
DFS — Depth-first search
UCS — Cost-based optimal search
A* — Heuristic + cost optimal search
